# Notebook 1 — Setup & Ingest Pipeline

*Hands-on time: ~40 minutes*

In this notebook you will:
1. Verify the environment (ES, Kibana, BIDS dataset)
2. Explore the BIDS dataset with `pybids`
3. Design an ElasticSearch index mapping
4. Generate text embeddings from metadata
5. Bulk-index everything into ES

**Prerequisites:** Complete the setup in `docs/00-overview.md` and read chapters 1–3.

---
## 1. Verify the Environment

In [ ]:
from elasticsearch import Elasticsearch

client = Elasticsearch("http://localhost:9200")

# Check connection
info = client.info()
print(f"Cluster name: {info['cluster_name']}")
print(f"ES version:   {info['version']['number']}")

health = client.cluster.health()
print(f"Status:        {health['status']}")
assert health['status'] in ('green', 'yellow'), "ES cluster is not healthy!"

In [ ]:
import os
from pathlib import Path

DATASET_DIR = Path("../data/ds001")
assert DATASET_DIR.exists(), f"Dataset not found at {DATASET_DIR.resolve()}. Run scripts/download_dataset.sh first."

print("Dataset contents:")
for item in sorted(DATASET_DIR.iterdir()):
    print(f"  {item.name}{'/' if item.is_dir() else ''}")

---
## 2. Explore the BIDS Dataset with pybids

`BIDSLayout` indexes the entire dataset and provides a query API that handles
the BIDS inheritance principle automatically.

In [ ]:
from bids import BIDSLayout

layout = BIDSLayout(str(DATASET_DIR), validate=False)

print(f"Subjects:    {layout.get_subjects()}")
print(f"Data types:  {layout.get_datatypes()}")
print(f"Suffixes:    {layout.get_suffixes()}")
print(f"Tasks:       {layout.get_tasks()}")
print(f"Total files: {len(layout.get(extension='.nii.gz'))} NIfTI images")

In [ ]:
# Look at one BOLD file's entities and metadata
bold_files = layout.get(suffix="bold", extension=".nii.gz")
example_file = bold_files[0]

print(f"File: {example_file.filename}")
print(f"\nEntities extracted from filename:")
for key, val in example_file.entities.items():
    print(f"  {key}: {val}")

print(f"\nResolved metadata from JSON sidecar(s):")
meta = layout.get_metadata(example_file.path)
for key, val in sorted(meta.items()):
    print(f"  {key}: {val}")

In [ ]:
import pandas as pd

# Read participant demographics
participants_file = DATASET_DIR / "participants.tsv"
if participants_file.exists():
    participants = pd.read_csv(participants_file, sep="\t")
    print("participants.tsv:")
    display(participants.head(10))
else:
    participants = pd.DataFrame()
    print("No participants.tsv found (optional in BIDS).")

---
## 3. Design the ElasticSearch Index Mapping

We define the schema explicitly so ES knows how to index each field optimally.
The mapping reflects the metadata fields we explored above.

In [ ]:
INDEX_NAME = "neuroimaging"
EMBEDDING_DIMS = 384  # all-MiniLM-L6-v2 output dimensionality

INDEX_MAPPING = {
    "mappings": {
        "properties": {
            # === Filename-derived entities ===
            "subject":   {"type": "keyword"},
            "session":   {"type": "keyword"},
            "task":      {"type": "keyword"},
            "run":       {"type": "keyword"},
            "suffix":    {"type": "keyword"},
            "datatype":  {"type": "keyword"},

            # === Participant demographics ===
            "age":       {"type": "float"},
            "sex":       {"type": "keyword"},

            # === JSON sidecar metadata (numeric) ===
            "RepetitionTime":        {"type": "float"},
            "EchoTime":              {"type": "float"},
            "FlipAngle":             {"type": "float"},
            "MagneticFieldStrength": {"type": "float"},
            "SliceThickness":        {"type": "float"},

            # === JSON sidecar metadata (categorical) ===
            "Manufacturer":           {"type": "keyword"},
            "ManufacturersModelName": {"type": "keyword"},
            "InstitutionName":        {"type": "keyword"},
            "PhaseEncodingDirection": {"type": "keyword"},

            # === JSON sidecar metadata (text — full-text searchable) ===
            "TaskName":          {"type": "text"},
            "SeriesDescription": {"type": "text"},

            # === Computed fields ===
            "description_text":   {"type": "text"},     # concatenated summary for BM25
            "metadata_embedding": {                       # dense vector for kNN
                "type": "dense_vector",
                "dims": EMBEDDING_DIMS,
                "similarity": "cosine",
                "index_options": {"type": "int8_hnsw"}
            },

            # === File reference ===
            "bids_path": {"type": "keyword"}
        }
    }
}

print("Mapping defined with fields:")
for field, config in INDEX_MAPPING["mappings"]["properties"].items():
    ftype = config.get("type", config.get("type", "?"))
    print(f"  {field:30s} → {ftype}")

In [ ]:
# Delete existing index if present (for re-runs), then create
if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'")

client.indices.create(index=INDEX_NAME, body=INDEX_MAPPING)
print(f"Created index '{INDEX_NAME}' with explicit mapping")

---
## 4. Generate Text Embeddings

We encode a concatenated metadata string for each scan. The embedding model
captures semantic relationships (e.g., \"T1w\" and \"structural MRI\" will be
close in vector space).

**First run:** The model (~80 MB) will be downloaded automatically.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded. Output dimensions: {model.get_sentence_embedding_dimension()}")

In [ ]:
def build_description_text(entities: dict, metadata: dict, participant_info: dict) -> str:
    """Concatenate key metadata into a human-readable string for embedding."""
    parts = []

    # Scan type
    suffix = entities.get("suffix", "")
    suffix_descriptions = {
        "T1w": "T1-weighted anatomical structural MRI",
        "T2w": "T2-weighted anatomical structural MRI",
        "bold": "BOLD functional MRI",
        "dwi": "diffusion weighted imaging",
        "FLAIR": "FLAIR MRI",
    }
    parts.append(suffix_descriptions.get(suffix, suffix))

    # Task
    task_name = metadata.get("TaskName", entities.get("task", ""))
    if task_name:
        parts.append(f"task: {task_name}")

    # Scanner
    field_strength = metadata.get("MagneticFieldStrength")
    if field_strength:
        parts.append(f"{field_strength}T")

    manufacturer = metadata.get("Manufacturer", "")
    model_name = metadata.get("ManufacturersModelName", "")
    if manufacturer:
        scanner = manufacturer
        if model_name:
            scanner += f" {model_name}"
        parts.append(scanner)

    # Timing parameters
    tr = metadata.get("RepetitionTime")
    if tr is not None:
        parts.append(f"RepetitionTime={tr}s")

    te = metadata.get("EchoTime")
    if te is not None:
        parts.append(f"EchoTime={te}s")

    fa = metadata.get("FlipAngle")
    if fa is not None:
        parts.append(f"FlipAngle={fa}deg")

    # Institution
    inst = metadata.get("InstitutionName", "")
    if inst:
        parts.append(f"Institution: {inst}")

    # Series description (free text from scanner)
    series_desc = metadata.get("SeriesDescription", "")
    if series_desc:
        parts.append(series_desc)

    # Demographics
    age = participant_info.get("age")
    sex = participant_info.get("sex")
    if age:
        parts.append(f"age {age}")
    if sex:
        parts.append(f"sex {sex}")

    return " | ".join(parts)


# Demonstrate with the example file from earlier
demo_entities = example_file.entities
demo_meta = layout.get_metadata(example_file.path)
demo_participant = {}
if not participants.empty:
    subj_id = f"sub-{demo_entities['subject']}"
    match = participants[participants["participant_id"] == subj_id]
    if not match.empty:
        demo_participant = match.iloc[0].to_dict()

demo_text = build_description_text(demo_entities, demo_meta, demo_participant)
print(f"Description text:\n  {demo_text}")

demo_embedding = model.encode(demo_text)
print(f"\nEmbedding shape: {demo_embedding.shape}")
print(f"First 10 values: {demo_embedding[:10]}")

---
## 5. Build & Run the Ingestion Pipeline

We iterate over every NIfTI file in the BIDS dataset, extract metadata,
generate embeddings, and bulk-index into ElasticSearch.

In [ ]:
from elasticsearch import helpers
from tqdm.notebook import tqdm
import numpy as np

# Build a lookup dict from participants.tsv
participant_lookup = {}
if not participants.empty:
    for _, row in participants.iterrows():
        participant_lookup[row["participant_id"]] = row.to_dict()


def generate_documents(layout, participant_lookup, model):
    """Yield ES documents for every NIfTI file in the BIDS layout."""
    nifti_files = layout.get(extension=".nii.gz")

    # Collect description texts for batch embedding
    file_data = []
    for bf in nifti_files:
        entities = bf.entities
        metadata = layout.get_metadata(bf.path)

        subj_id = f"sub-{entities.get('subject', '')}"
        participant_info = participant_lookup.get(subj_id, {})

        desc_text = build_description_text(entities, metadata, participant_info)

        file_data.append({
            "entities": entities,
            "metadata": metadata,
            "participant_info": participant_info,
            "desc_text": desc_text,
            "bids_path": str(Path(bf.path).relative_to(DATASET_DIR.resolve())),
        })

    # Batch-encode all description texts (much faster than one-by-one)
    desc_texts = [fd["desc_text"] for fd in file_data]
    print(f"Encoding {len(desc_texts)} descriptions...")
    embeddings = model.encode(desc_texts, show_progress_bar=True, batch_size=32)

    # Yield ES documents
    for fd, embedding in zip(file_data, embeddings):
        entities = fd["entities"]
        metadata = fd["metadata"]
        participant_info = fd["participant_info"]

        doc = {
            # Filename entities
            "subject":   str(entities.get("subject", "")),
            "session":   str(entities.get("session", "")),
            "task":      str(entities.get("task", "")),
            "run":       str(entities.get("run", "")),
            "suffix":    str(entities.get("suffix", "")),
            "datatype":  str(entities.get("datatype", "")),

            # Demographics
            "age":       participant_info.get("age"),
            "sex":       str(participant_info.get("sex", "")),

            # Sidecar metadata (numeric)
            "RepetitionTime":        metadata.get("RepetitionTime"),
            "EchoTime":              metadata.get("EchoTime"),
            "FlipAngle":             metadata.get("FlipAngle"),
            "MagneticFieldStrength": metadata.get("MagneticFieldStrength"),
            "SliceThickness":        metadata.get("SliceThickness"),

            # Sidecar metadata (categorical)
            "Manufacturer":           metadata.get("Manufacturer"),
            "ManufacturersModelName": metadata.get("ManufacturersModelName"),
            "InstitutionName":        metadata.get("InstitutionName"),
            "PhaseEncodingDirection": metadata.get("PhaseEncodingDirection"),

            # Sidecar metadata (text)
            "TaskName":          metadata.get("TaskName"),
            "SeriesDescription": metadata.get("SeriesDescription"),

            # Computed
            "description_text":   fd["desc_text"],
            "metadata_embedding": embedding.tolist(),

            # File path
            "bids_path": fd["bids_path"],
        }

        yield {"_index": INDEX_NAME, "_source": doc}


print(f"Starting ingestion into index '{INDEX_NAME}'...")

In [ ]:
# Run the bulk ingestion
success_count, errors = helpers.bulk(
    client,
    generate_documents(layout, participant_lookup, model),
    raise_on_error=True,
    refresh="wait_for"  # ensure docs are searchable immediately after
)

print(f"\nIngestion complete!")
print(f"  Documents indexed: {success_count}")
print(f"  Errors:            {len(errors) if isinstance(errors, list) else errors}")

---
## 6. Verify the Index

In [ ]:
# Count documents
count = client.count(index=INDEX_NAME)
print(f"Total documents in '{INDEX_NAME}': {count['count']}")
assert count['count'] > 0, "No documents were indexed!"

# Retrieve one document to inspect
sample = client.search(
    index=INDEX_NAME,
    body={"query": {"match_all": {}}, "size": 1},
    fields=["subject", "suffix", "task", "RepetitionTime", "Manufacturer", "description_text", "bids_path"]
)

hit = sample["hits"]["hits"][0]
print(f"\nSample document (fields):")
for field, values in hit.get("fields", {}).items():
    print(f"  {field}: {values}")

In [ ]:
# Quick aggregation: how many documents per suffix?
agg_result = client.search(
    index=INDEX_NAME,
    body={
        "size": 0,
        "aggs": {
            "by_suffix": {
                "terms": {"field": "suffix"}
            }
        }
    }
)

print("Documents by suffix:")
for bucket in agg_result["aggregations"]["by_suffix"]["buckets"]:
    print(f"  {bucket['key']:15s} {bucket['doc_count']:4d} docs")

---
## Summary

You've now:
- ✅ Connected to ElasticSearch and verified the cluster
- ✅ Explored a BIDS dataset programmatically with `pybids`
- ✅ Created an ES index with explicit mappings for structured metadata + dense vectors
- ✅ Generated embeddings from concatenated metadata using `sentence-transformers`
- ✅ Bulk-indexed all documents into ES
- ✅ Verified the index with counts and aggregations

**Next:** Open **Notebook 2** to start querying this data with keyword and filtered search.